# Chapter 7: Valuation of Swaps
## Replicating John Hull's Core Valuation Models & Practice Questions with Python

Imagine you and a friend are at a coffee shop. You have a variable-rate bank loan, and you are constantly stressing about whether interest rates will spike next month. Your friend has a fixed-rate loan and is perfectly relaxed, but secretly wishes they could capture lower payments if rates drop. You decide to make a deal: you'll pay your friend a steady, predictable interest rate, and in return, they'll pay you whatever the floating rate is. You've just entered into an **Interest Rate Swap**!

The principal amount of your loans never actually changes hands—that's the **Notional Principal**. You only exchange the *difference* in the interest payments.

In this notebook, we are going to dive deep into **Chapter 7** of John Hull's classic *Options, Futures, and Other Derivatives*. We will replicate the core swap valuation models step-by-step in Python:
1. **Example 7.1**: Valuing an **Interest Rate Swap** as a portfolio of Forward Rate Agreements (FRAs).
2. **Example 7.2**: Valuing a **Fixed-for-Fixed Currency Swap** as a portfolio of forward foreign exchange contracts.
3. **Example 7.3**: Valuing that same **Currency Swap** as the difference between two bonds (Yen and Dollar).
4. **Practice Questions**: Solving and programming the math for **Practice Questions 7.1 to 7.23**.

---

# Part 1: Valuation of Interest Rate Swaps (Example 7.1)

### The Setup: Example 7.1

A financial institution entered into a swap some time ago with a notional principal of **$100 million**. 
* **What they pay (Fixed Leg)**: Semiannual interest at a rate of **$3.0\%$** per annum.
* **What they receive (Floating Leg)**: The 6-month SOFR reference rate.
* **Remaining life of the swap**: **$1.2\text{ years}$**. 
* **When the cash exchanges happen**: At $0.2$, $0.7$, and $1.2$ years from today.

#### The Market Data we have today:
1. **Risk-free Rates (Continuous Compounding)**:
 * For a 0.2-year maturity: $2.8\%$
 * For a 0.7-year maturity: $3.2\%$
 * For a 1.2-year maturity: $3.4\%$
2. **Observed Floating Rate for the First Period ($0.0 \to 0.2$ years)**:
 * The floating rate for the first payment at $0.2$ years was actually locked in some time ago. The continuously compounded rate observed for the last $0.3$ years (which covers $60\%$ of the period determining this payment) is $2.3\%$.
 * Taking the weighted average, the floating rate for the exchange at $0.2$ years is:
 $$R_c(0.2) = 0.6 \times 2.3\% + 0.4 \times 2.8\% = 2.50\% \text{ (continuous compounding)}$$


---
## The Core Methodology: Section 7.6 (Valuation of Interest Rate Swaps)

Before diving into the calculations, let's ground our work in the fundamental theory. In Section 7.6, John Hull outlines the core mechanics of swap valuation:

> [!NOTE]
> ### From John Hull (Section 7.6: Valuation of Interest Rate Swaps):
> 
> *"An interest rate swap is worth close to zero when it is first initiated. After it has been in existence for some time, its value may be positive or negative.*
> 
> *Each exchange of payment in a swap can be regarded as a forward rate agreement (FRA). As shown at the end of Section 4.9, an FRA can be valued by assuming that forward rates are realized. Because it is nothing more than a portfolio of FRAs, an interest rate swap can also be valued by assuming that forward rates are realized. The procedure is:*
> 
> 1. *Calculate forward rates for each of the unknown floating rates that will determine swap cash flows.*
> 2. *Calculate the swap cash flows on the assumption that unknown floating rates will equal forward rates.*
> 3. *Discount the swap cash flows at the risk-free rate.*
> 
> *... Note that the first rate required is not really a forward rate. It is a blend of one-day rates already observed and the zero rate that corresponds to the remaining one-day rates in the period."*

Let's unpack three crucial takeaways from Hull's explanation:
1. **Initial Value is Zero**: At inception, fixed rates are set so that the floating leg and the fixed leg balance perfectly ($V_{\text{swap}} \approx 0$). No money changes hands at day one. Over time, as interest rates move, the swap gains a positive or negative value, shifting wealth between the counterparties.
2. **The Portfolio of FRAs**: Each future payment exchange is treated as an independent **Forward Rate Agreement (FRA)**. By calculating the implied forward rates for these unknown periods, we can project future cash flows as if they are certain, and then discount them back.
3. **The First "Blended" Rate**: Why do we use $2.50\%$ for the first payment? Because that 0.2-year period is already partially completed. Part of the floating rate has already been observed ($2.3\%$ over the last $0.3$ years), and the rest is determined by the current $0.2$-year spot rate ($2.8\%$). Hence, it is a **blend** rather than a pure forward rate.


---
## The Math Behind the Swap: Intuitions and Formulas

Valuing a swap might seem intimidating, but it boils down to answering two simple questions:
1. **What cash flows do we expect to pay and receive in the future?**
2. **What are those future cash flows worth to us today?**

Let's break down the mathematical steps with some core financial intuition.

### Step 1: Predicting Future Floating Rates (Forward Rates)
We know what we will receive at $0.2$ years ($2.50\%$), but what about the payments at $0.7$ and $1.2$ years? Since we don't have a time machine to see future SOFR rates, we assume that the market's **forward rates** will be realized.

Think of it like renting a car. If a 1-day rental costs $50, and a 2-day rental costs $90, the "implied rent" for the second day alone is $40. Under continuous compounding, the forward rate $F$ between two future times $T_1$ and $T_2$ is calculated similarly:
$$F = \frac{r_2 T_2 - r_1 T_1}{T_2 - T_1}$$

Using our zero rates:
* **For the period between 0.2 and 0.7 years**:
 $$F(0.2, 0.7) = \frac{0.032 \times 0.7 - 0.028 \times 0.2}{0.7 - 0.2} = 3.36\%$$
* **For the period between 0.7 and 1.2 years**:
 $$F(0.7, 1.2) = \frac{0.034 \times 1.2 - 0.032 \times 0.7}{1.2 - 0.7} = 3.68\%$$

### Step 2: Adjusting for Compounding Frequency
The interest rates we observe in the market are usually expressed with **continuous compounding** (the mathematical ideal where interest compounds every split-second). However, swap payments are actually exchanged **semiannually** (discrete compounding twice a year, or $\Delta t = 0.5$ years).

Imagine a snowball rolling down a hill. Continuous compounding is a smooth, uninterrupted roll where the snowball grows at every millimeter. Semiannual compounding is like the snowball only gathering snow at two specific checkpoints. To end up with the same final size, the semiannual snowball needs a slightly harder push (a numerically higher nominal rate $R_m$) than the continuous snowball ($R_c$). 

We convert the continuous rates to their semiannual equivalents using:
$$e^{R_c \times 0.5} = \left(1 + \frac{R_m}{2}\right)^{2 \times 0.5} \implies R_m = 2 \left(e^{R_c / 2} - 1\right)$$

### Step 3: Calculating Cash Flows
For each exchange date $t_i$:
* **Fixed Cash Flow**: We make a fixed payment. Since the payment covers half a year ($0.5$ years) at a $3\%$ fixed rate:
 $$\text{Fixed Cash Flow} = -0.5 \times 0.03 \times \$100\text{M} = -\$1.500\text{ million}$$
* **Floating Cash Flow**: We receive the semiannual floating rate $R_m$ on our principal:
 $$\text{Floating Cash Flow} = 0.5 \times R_m \times \$100\text{M}$$
* **Net Cash Flow**: The difference between what we receive and what we pay:
 $$\text{Net Cash Flow} = \text{Floating Cash Flow} + \text{Fixed Cash Flow}$$

### Step 4: Discounting back to Today
Finally, money tomorrow is worth less than money today. We pull each net cash flow back to the present using the zero rate $r(t_i)$ as our discount factor:
$$\text{Discount Factor } (DF_i) = e^{-r_t \times t_i}$$
$$\text{Present Value } (PV_i) = \text{Net Cash Flow}_i \times DF_i$$

Let's plug these steps into Python!


In [5]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Format pandas decimals to match the book's precision
pd.options.display.float_format = '{:,.4f}'.format

# --- Example 7.1 Setup Variables ---
notional_ir = 100.0 # $100 Million
fixed_rate_ir = 0.03 # 3.0% per annum
times_ir = np.array([0.2, 0.7, 1.2]) # Payment dates (years)
zero_rates_ir = np.array([0.028, 0.032, 0.034]) # Continuous risk-free rates


In [6]:
# --- Step 1: Calculate Continuous Floating Rates (Forward Rates) ---
rc_ir_02 = 0.6 * 0.023 + 0.4 * 0.028 # First payment blend
rc_ir_07 = (zero_rates_ir[1] * times_ir[1] - zero_rates_ir[0] * times_ir[0]) / (times_ir[1] - times_ir[0])
rc_ir_12 = (zero_rates_ir[2] * times_ir[2] - zero_rates_ir[1] * times_ir[1]) / (times_ir[2] - times_ir[1])
rc_rates_ir = np.array([rc_ir_02, rc_ir_07, rc_ir_12])

# --- Step 2: Convert to Semiannual Compounding Equivalents ---
rm_rates_ir = 2 * (np.exp(rc_rates_ir / 2) - 1)

# Inspect the continuous-to-semiannual conversion premium
print("Interest Rate Swap Term Structure:")
for t, rc, rm in zip(times_ir, rc_rates_ir, rm_rates_ir):
 print(f" Time {t}y -> Continuous Floating Rate: {rc*100:.3f}% | Semiannual Compounding: {rm*100:.3f}%")


Interest Rate Swap Term Structure:
  Time 0.2y -> Continuous Floating Rate: 2.500% | Semiannual Compounding: 2.516%
  Time 0.7y -> Continuous Floating Rate: 3.360% | Semiannual Compounding: 3.388%
  Time 1.2y -> Continuous Floating Rate: 3.680% | Semiannual Compounding: 3.714%


In [7]:
# --- Step 3 & 4: Compute Cash Flows and Present Values ---
fixed_cfs_ir = -0.5 * fixed_rate_ir * notional_ir * np.ones_like(times_ir)
floating_cfs_ir = 0.5 * rm_rates_ir * notional_ir
net_cfs_ir = fixed_cfs_ir + floating_cfs_ir

discount_factors_ir = np.exp(-zero_rates_ir * times_ir)
pvs_ir = net_cfs_ir * discount_factors_ir

# Assemble valuation table
df_swap_ir = pd.DataFrame({
 'Time (years)': times_ir,
 'Fixed cash flow': fixed_cfs_ir,
 'Floating cash flow': floating_cfs_ir,
 'Net cash flow': net_cfs_ir,
 'Discount factor': discount_factors_ir,
 'Present value of net CF': pvs_ir
})

# Add a summary row
df_total_ir = pd.DataFrame({
 'Time (years)': ['Total'],
 'Fixed cash flow': [fixed_cfs_ir.sum()],
 'Floating cash flow': [floating_cfs_ir.sum()],
 'Net cash flow': [net_cfs_ir.sum()],
 'Discount factor': [np.nan],
 'Present value of net CF': [pvs_ir.sum()]
})

df_valuation_ir = pd.concat([df_swap_ir, df_total_ir], ignore_index=True)
df_valuation_ir.set_index('Time (years)', inplace=True)
df_valuation_ir


,Fixed cash flow,Floating cash flow,Net cash flow,Discount factor,Present value of net CF
Time (years),,,,,
0.2000,-1.5000,1.2578,-0.2422,0.9944,-0.2408
0.7000,-1.5000,1.6942,0.1942,0.9778,0.1899
1.2000,-1.5000,1.8570,0.3570,0.9600,0.3428
Total,-4.5000,4.8091,0.3091,NaN,0.2918


---
## Discussion of Example 7.1

Our Python results perfectly match John Hull's textbook table:
* **Total Swap Value**: **$+\$0.292\text{ million}$** ($0.2918$ million in our exact calculation).
* Since the value is **positive**, the swap is an asset to our financial institution because our incoming floating leg pays more than our fixed obligation.


---
# Part 2: Valuation of Fixed-for-Fixed Currency Swaps (Examples 7.2 & 7.3)

Now imagine a new scenario. Your coffee shop friend lives in Tokyo and runs a high-end restaurant. You are running a tech startup in New York. You need Japanese Yen for your local operations in Tokyo, and your friend needs US Dollars to purchase equipment.

To avoid bank conversion fees and borrow at better rates locally, you decide to enter into a **Fixed-for-Fixed Currency Swap**:
* **Initial Exchange**: You give your friend **$10 million** today, and in return he gives you **1,200 million Yen** (locking in a spot rate of $110\text{ Yen per Dollar}$).
* **Annual Exchange (The Legs)**: 
 * You pay your friend a **$4.0\%$** fixed rate per annum in **Dollars**.
 * Your friend pays you a **$3.0\%$** fixed rate per annum in **Yen**.
* **Final Exchange**: At the end of the swap, you exchange the principals back (you return the 1,200 million Yen principal, and he returns the $10 million principal).

---

### The Setup: Examples 7.2 & 7.3

* **Remaining Life**: **$3\text{ years}$**. Payments occur annually (at $t=1$, $t=2$, and $t=3$ years).
* **Current Spot Exchange Rate ($S_0$)**: $110\text{ Yen per Dollar}$ (or $1/110 = 0.009091$ USD per JPY).
* **Yield Curves (Flat)**:
 * **U.S. risk-free rate ($r$)**: $2.5\%$ per annum (continuous compounding).
 * **Japanese risk-free rate ($r_f$)**: $1.5\%$ per annum (continuous compounding).

We will value this swap using **two completely different but theoretically equivalent methods**:
1. **Method 1 (Example 7.2)**: Valuing the swap as a portfolio of **Forward Foreign Exchange Contracts**.
2. **Method 2 (Example 7.3)**: Valuing the swap as the difference between **Two Fixed-Rate Bonds** (a Dollar-denominated bond and a Yen-denominated bond).


---
## Method 1: Portfolio of Forward Contracts (Example 7.2)

Under Hull's first approach, each annual payment exchange is a **Forward Contract**. 
To value these forward contracts, we calculate the implied **forward exchange rate** for each year, determine the net dollar cash flows if these rates are realized, and discount them back to today at the domestic (U.S.) risk-free rate.

### Step 1: Calculate Forward Exchange Rates
Using the interest rate parity equation, the forward exchange rate $F_t$ (in Dollars per Yen) for maturity $t$ is:
$$F_t = S_0 e^{(r - r_f) \times t}$$

Where:
* $S_0 = 1/110 = 0.009091\text{ USD/JPY}$
* $r = 0.025$ (U.S. rate)
* $r_f = 0.015$ (Japanese rate)

* **Year 1 Forward**: $0.009091 e^{(0.025 - 0.015) \times 1} = 0.009182$
* **Year 2 Forward**: $0.009091 e^{(0.025 - 0.015) \times 2} = 0.009275$
* **Year 3 Forward**: $0.009091 e^{(0.025 - 0.015) \times 3} = 0.009368$

### Step 2: Project Cash Flows (in Millions)
* **Dollar Payments**: $4\% \times \$10\text{M} = -\$0.4\text{M}$ per year. In Year 3, we also pay back the principal: $-\$10.4\text{M}$.
* **Yen Receipts**: $3\% \times 1,200\text{M JPY} = +36\text{M JPY}$ per year. In Year 3, we receive the JPY principal: $+1,236\text{M JPY}$.

### Step 3: Compute Net Dollar Flows & Discount
For each year $t$:
1. Convert JPY cash flows to USD using $F_t$.
2. Compute the net cash flow in Dollars: $\text{Net USD CF}_t = (\text{Yen CF}_t \times F_t) + \text{Dollar CF}_t$.
3. Discount the net flow at the U.S. rate: $\text{PV}_t = \text{Net USD CF}_t \times e^{-r \times t}$.


In [8]:
# --- Example 7.2: Forward Contract Method ---
usd_principal = 10.0 # $10 Million
jpy_principal = 1200.0 # 1200 Million Yen
usd_rate = 0.04 # 4.0% interest paid
jpy_rate = 0.03 # 3.0% interest received

r_usd = 0.025 # U.S. risk-free rate
r_jpy = 0.015 # Japan risk-free rate
spot_s0 = 1.0 / 110.0 # Spot Exchange Rate (USD per JPY)
times_curr = np.array([1.0, 2.0, 3.0]) # 3 Years remaining

# 1. Project cash flows
usd_cfs = -usd_rate * usd_principal * np.ones_like(times_curr)
usd_cfs[-1] -= usd_principal # Pay USD Principal in Year 3

jpy_cfs = jpy_rate * jpy_principal * np.ones_like(times_curr)
jpy_cfs[-1] += jpy_principal # Receive JPY Principal in Year 3

# 2. Calculate Forward exchange rates
forward_rates = spot_s0 * np.exp((r_usd - r_jpy) * times_curr)

# 3. Convert JPY flows to USD and find Net Flows
usd_val_of_jpy = jpy_cfs * forward_rates
net_usd_cfs = usd_val_of_jpy + usd_cfs

# 4. Discount back at U.S. risk-free rate
discount_factors_usd = np.exp(-r_usd * times_curr)
pvs_curr_fwd = net_usd_cfs * discount_factors_usd

# Create valuation table
df_swap_fwd = pd.DataFrame({
 'Time (years)': times_curr,
 'Dollar cash flow': usd_cfs,
 'Yen cash flow': jpy_cfs,
 'Forward exchange rate': forward_rates,
 'Dollar value of yen CF': usd_val_of_jpy,
 'Net cash flow': net_usd_cfs,
 'Present value': pvs_curr_fwd
})

df_total_fwd = pd.DataFrame({
 'Time (years)': ['Total'],
 'Dollar cash flow': [usd_cfs.sum()],
 'Yen cash flow': [jpy_cfs.sum()],
 'Forward exchange rate': [np.nan],
 'Dollar value of yen CF': [usd_val_of_jpy.sum()],
 'Net cash flow': [net_usd_cfs.sum()],
 'Present value': [pvs_curr_fwd.sum()]
})

df_valuation_fwd = pd.concat([df_swap_fwd, df_total_fwd], ignore_index=True)
df_valuation_fwd.set_index('Time (years)', inplace=True)
df_valuation_fwd


,Dollar cash flow,Yen cash flow,Forward exchange rate,Dollar value of yen CF,Net cash flow,Present value
Time (years),,,,,,
1.0000,-0.4000,36.0000,0.0092,0.3306,-0.0694,-0.0677
2.0000,-0.4000,36.0000,0.0093,0.3339,-0.0661,-0.0629
3.0000,-10.4000,"1,236.0000",0.0094,11.5786,1.1786,1.0934
Total,-11.2000,"1,308.0000",NaN,12.2430,1.0430,0.9628


---
## Method 2: Valuation in Terms of Bond Prices (Example 7.3)

Rather than checking forward exchange rates, we can think of the currency swap as **two separate bonds**:
1. We are long a **Yen-denominated bond** paying a $3\%$ annual coupon and a $1,200\text{M JPY}$ principal.
2. We are short a **Dollar-denominated bond** paying a $4\%$ annual coupon and a $\$10\text{M USD}$ principal.

The value of the swap in Dollars is simply:
$$V_{\text{swap}} = S_0 B_{\text{Yen}} - B_{\text{Dollar}}$$

Where:
* $B_{\text{Yen}}$ is the value of the Yen bond in JPY (discounted at the Japanese risk-free rate $1.5\%$).
* $B_{\text{Dollar}}$ is the value of the Dollar bond in USD (discounted at the U.S. risk-free rate $2.5\%$).
* $S_0$ is the current spot exchange rate ($1/110$ USD per JPY).


In [9]:
# --- Example 7.3: Bond Valuation Method ---

# 1. Dollar Bond Cash Flows
bond_usd_cfs = usd_rate * usd_principal * np.ones_like(times_curr)
bond_usd_cfs[-1] += usd_principal

df_usd_bond = np.exp(-r_usd * times_curr)
pv_usd_bond_cfs = bond_usd_cfs * df_usd_bond
b_usd = pv_usd_bond_cfs.sum()

# 2. Yen Bond Cash Flows
bond_jpy_cfs = jpy_rate * jpy_principal * np.ones_like(times_curr)
bond_jpy_cfs[-1] += jpy_principal

df_jpy_bond = np.exp(-r_jpy * times_curr)
pv_jpy_bond_cfs = bond_jpy_cfs * df_jpy_bond
b_jpy = pv_jpy_bond_cfs.sum()

# 3. Assemble Bond Tables
df_bonds = pd.DataFrame({
 'Time (years)': times_curr,
 'Cash flows on dollar bond ($)': bond_usd_cfs,
 'Present value ($)': pv_usd_bond_cfs,
 'Cash flows on yen bond (yen)': bond_jpy_cfs,
 'Present value (yen)': pv_jpy_bond_cfs
})

df_bonds_total = pd.DataFrame({
 'Time (years)': ['Total'],
 'Cash flows on dollar bond ($)': [bond_usd_cfs.sum()],
 'Present value ($)': [b_usd],
 'Cash flows on yen bond (yen)': [bond_jpy_cfs.sum()],
 'Present value (yen)': [b_jpy]
})

df_valuation_bonds = pd.concat([df_bonds, df_bonds_total], ignore_index=True)
df_valuation_bonds.set_index('Time (years)', inplace=True)

# 4. Calculate Net Swap Value in Dollars
v_swap_dollars = (b_jpy * spot_s0) - b_usd

print(df_valuation_bonds.to_string())
print(f"\nValue of Dollar Bond (B_D): ${b_usd:.4f} Million USD")
print(f"Value of Yen Bond (B_F): {b_jpy:.4f} Million Yen")
print(f"Value of Swap (V_swap): ${v_swap_dollars:.4f} Million USD")


              Cash flows on dollar bond ($)  Present value ($)  Cash flows on yen bond (yen)  Present value (yen)
Time (years)                                                                                                     
1.0000                               0.4000             0.3901                       36.0000              35.4640
2.0000                               0.4000             0.3805                       36.0000              34.9360
3.0000                              10.4000             9.6485                    1,236.0000           1,181.6129
 Total                              11.2000            10.4191                    1,308.0000           1,252.0130

Value of Dollar Bond (B_D): $10.4191 Million USD
Value of Yen Bond (B_F): 1252.0130 Million Yen
Value of Swap (V_swap): $0.9628 Million USD


---
## Discussion & Financial Takeaways (Currency Swaps)

Both valuation methods yield the exact same value of **$+\$0.9629\text{ million}$**!

### Key Insight: The High vs. Low Interest Rate Currencies
Notice the net cash flow direction for the early years versus the final year under the Forward Contract approach:
* **Year 1 PV**: $-\$0.0677\text{ million}$
* **Year 2 PV**: $-\$0.0629\text{ million}$
* **Year 3 PV**: $+\$1.0934\text{ million}$

Why does this asymmetrical pattern happen? 
Since interest rates in the U.S. ($2.5\%$) are higher than in Japan ($1.5\%$), the U.S. Dollar trades at a **forward discount** relative to the Yen. In other words, Jpy is expected to strengthen over time in Dollar terms.

* **Early Years**: The interest we receive in Jpy is converted at forward rates that do not fully offset the larger interest payment we make in USD. The net annual cash flows are **negative** ($-\$0.0694\text{M}$ and $-\$0.0661\text{M}$).
* **Year 3 (Principal Exchange)**: Jpy has appreciated so much in forward terms that returning the Japanese principal yields a massive positive net cash flow of **$+\$1.1786\text{M}$** in USD, easily eclipsing the negative early payments.

As Hull highlights:
* **For the payer of the high-interest currency (USD)**: The early cash flows have a negative value, but the final principal exchange has a strong positive value. The swap value tends to be positive during most of its life.
* **For the payer of the low-interest currency (JPY)**: The exact opposite holds. The early exchanges are positive, but the final principal exchange is highly negative. This swap tends to have a negative value for most of its life.

Understanding this asymmetry is crucial when financial institutions manage **credit risk** in bilateral transactions, as the party holding a positive swap value carries exposure to the counterparty's default.


---
# Part 3: Chapter 7 Practice Questions & Python Solutions

In this section, we present comprehensive, friendly, and precise solutions to **Practice Questions 7.1 to 7.23** from John Hull's Chapter 7. 

For all quantitative questions, we implement the math directly in Python so you can run the cells and see the exact, rounded outcomes!


---
## Practice Question 7.1

### Question
Companies A and B have been offered the following rates per annum on a $20 million five-year loan:
* **Company A**: Fixed $5.0\%$ | Floating $\text{SOFR} + 0.1\%$
* **Company B**: Fixed $6.4\%$ | Floating $\text{SOFR} + 0.6\%$

Company A requires a floating-rate loan; Company B requires a fixed-rate loan. Design a swap that will net a bank, acting as intermediary, $0.1\%$ per annum and that will appear equally attractive to both companies.

### Solution
1. **Identify Spreads**:
 * Fixed rate spread: $6.4\% - 5.0\% = 1.4\%$ (Company A has a $1.4\%$ advantage).
 * Floating rate spread: $(\text{SOFR} + 0.6\%) - (\text{SOFR} + 0.1\%) = 0.5\%$ (Company A has a $0.5\%$ advantage).
 * Total potential gain from swap: $\text{Fixed spread} - \text{Floating spread} = 1.4\% - 0.5\% = 0.9\%$ per annum.
2. **Allocate Gains**:
 * The intermediary bank receives $0.1\%$, leaving $0.9\% - 0.1\% = 0.8\%$ per annum to be split between the companies.
 * To be equally attractive, each company must save $0.4\%$ relative to their market quotes.
3. **Determine Target Costs**:
 * **Company A** wants floating: Offered $\text{SOFR} + 0.1\%$ in the market. With $0.4\%$ savings, target cost is **$\text{SOFR} - 0.3\%$**.
 * **Company B** wants fixed: Offered $6.4\%$ in the market. With $0.4\%$ savings, target cost is **$6.0\%$**.
4. **Swap Design**:
 * **Company A** borrows at fixed $5.0\%$ in the market.
 * Pays fixed $5.0\%$ to the market.
 * Pays $\text{SOFR}$ to the bank.
 * Receives fixed $5.3\%$ from the bank.
 * **Net Cost for A**: $5.0\% + \text{SOFR} - 5.3\% = \mathbf{\text{SOFR} - 0.3\%}$ (Savings of $0.4\%$, matches target!).
 * **Company B** borrows at floating $\text{SOFR} + 0.6\%$ in the market.
 * Pays floating $\text{SOFR} + 0.6\%$ to the market.
 * Pays fixed $5.4\%$ to the bank.
 * Receives $\text{SOFR}$ from the bank.
 * **Net Cost for B**: $(\text{SOFR} + 0.6\%) + 5.4\% - \text{SOFR} = \mathbf{6.0\%}$ fixed (Savings of $0.4\%$, matches target!).
 * **The Bank** (Intermediary):
 * Receives $\text{SOFR}$ from A and pays $\text{SOFR}$ to B (net floating flow is 0).
 * Receives fixed $5.4\%$ from B and pays fixed $5.3\%$ to A.
 * **Bank Spread**: $5.4\% - 5.3\% = \mathbf{0.1\%}$ per annum (Matches target!).


---
## Practice Question 7.2

### Question
A $100 million interest rate swap has a remaining life of 10 months. Under the terms of the swap, six-month LIBOR is exchanged for 4% per annum (compounded semiannually). Six-month LIBOR forward rates for all maturities are 3% (with semiannual compounding). The six-month LIBOR rate was 2.4% two months ago. OIS rates for all maturities are 2.7% with continuous compounding. What is the current value of the swap to the party paying floating? What is the value to the party paying fixed?

### Solution
Since the swap payments are semiannual, and the remaining life is 10 months, payment exchanges occur in **4 months** ($0.3333$ years) and **10 months** ($0.8333$ years).

* **Payment at 4 Months (0.3333y)**:
 * This floating rate was set 2 months ago (at the start of the 6-month period). The 6-month LIBOR rate then was $2.4\%$.
 * Floating CF paid: $0.5 \times 0.024 \times \$100\text{M} = \$1.2\text{M}$.
 * Fixed CF received: $0.5 \times 0.04 \times \$100\text{M} = \$2.0\text{M}$.
* **Payment at 10 Months (0.8333y)**:
 * This floating rate is unknown today. We assume the forward rate of $3.0\%$ is realized.
 * Floating CF paid: $0.5 \times 0.030 \times \$100\text{M} = \$1.5\text{M}$.
 * Fixed CF received: $0.5 \times 0.04 \times \$100\text{M} = \$2.0\text{M}$.
* **Discounting**:
 * OIS continuous rate is flat at $2.7\%$ per annum.
 * $DF_1 = e^{-0.027 \times 4/12}$
 * $DF_2 = e^{-0.027 \times 10/12}$


In [10]:
# --- Question 7.2 Calculations ---
notional_72 = 100.0 # $100 Million
fixed_rate_72 = 0.04
libor_first_72 = 0.024
libor_fwd_72 = 0.030
r_ois_72 = 0.027 # Continuous

times_72 = np.array([4.0/12.0, 10.0/12.0])

# Cash flows (receives fixed, pays floating)
fixed_cfs_72 = 0.5 * fixed_rate_72 * notional_72 * np.ones_like(times_72)
floating_cfs_72 = 0.5 * np.array([libor_first_72, libor_fwd_72]) * notional_72
net_cfs_72 = fixed_cfs_72 - floating_cfs_72 # Pay Floating, Receive Fixed

dfs_72 = np.exp(-r_ois_72 * times_72)
pvs_72 = net_cfs_72 * dfs_72
v_pay_float = pvs_72.sum()

print(f"Value to the party paying floating (receives fixed): +${v_pay_float:.4f} Million")
print(f"Value to the party paying fixed (receives floating): -${v_pay_float:.4f} Million")


Value to the party paying floating (receives fixed): +$1.2817 Million
Value to the party paying fixed (receives floating): -$1.2817 Million


---
## Practice Question 7.3

### Question
Company X wishes to borrow U.S. dollars at a fixed rate of interest. Company Y wishes to borrow Japanese yen at a fixed rate of interest. The amounts required by the two companies are roughly the same at the current exchange rate. The companies have been quoted the following interest rates, which have been adjusted for the impact of taxes:
* **Company X**: Yen $5.0\%$ | Dollars $9.6\%$
* **Company Y**: Yen $6.5\%$ | Dollars $10.0\%$

Design a swap that will net a bank, acting as intermediary, 50 basis points per annum. Make the swap equally attractive to the two companies and ensure that all foreign exchange risk is assumed by the bank.

### Solution
1. **Identify Spreads**:
 * Yen spread: $6.5\% - 5.0\% = 1.5\%$ (X has a $1.5\%$ advantage).
 * Dollar spread: $10.0\% - 9.6\% = 0.4\%$ (X has a $0.4\%$ advantage).
 * Total gain from swap: $\text{Yen spread} - \text{Dollar spread} = 1.5\% - 0.4\% = 1.1\%$ per annum.
2. **Allocate Gains**:
 * The intermediary bank receives $50\text{ bps} = 0.5\%$, leaving $1.1\% - 0.5\% = 0.6\%$ per annum.
 * Split equally, each company saves $0.3\%$ relative to market quotes.
3. **Determine Target Costs**:
 * **Company X** wants USD: Offered $9.6\%$. With $0.3\%$ savings, target cost is **$9.3\%$** fixed in USD.
 * **Company Y** wants Yen: Offered $6.5\%$. With $0.3\%$ savings, target cost is **$6.2\%$** fixed in JPY.
4. **Swap Design**:
 * **Company X** borrows Yen in market at $5.0\%$.
 * Pays JPY fixed $5.0\%$ to market.
 * Pays USD fixed $9.3\%$ to bank.
 * Receives JPY fixed $5.0\%$ from bank.
 * **Net USD Cost to X**: $9.3\%$ USD fixed (Savings of $0.3\%$, matches target!).
 * **Company Y** borrows Dollars in market at $10.0\%$.
 * Pays USD fixed $10.0\%$ to market.
 * Pays JPY fixed $6.2\%$ to bank.
 * Receives USD fixed $10.0\%$ from bank.
 * **Net JPY Cost to Y**: $6.2\%$ JPY fixed (Savings of $0.3\%$, matches target!).
 * **The Bank** (Intermediary):
 * Dollar flows: Receives $9.3\%$ from X, pays $10.0\%$ to Y. **Net USD Flow**: $-0.7\%$ per annum (Bank pays $0.7\%$ USD).
 * Yen flows: Receives $6.2\%$ from Y, pays $5.0\%$ to X. **Net JPY Flow**: $+1.2\%$ per annum (Bank receives $1.2\%$ JPY).
 * Let $S_0$ be the exchange rate. The net spread to the bank is $+1.2\%\text{ JPY} - 0.7\%\text{ USD}$.
 * Since the Yen principal and Dollar principal are equivalent at the spot exchange rate, the value of the $1.2\%$ Yen receipt corresponds to a $1.2\%$ Dollar value at inception. The bank's net spread is $1.2\% - 0.7\% = 0.5\% = 50\text{ basis points}$. 
 * Since the bank receives Jpy and pays Dollars, it assumes the **foreign exchange risk** as requested.


---
## Practice Question 7.4

### Question
A currency swap has a remaining life of 15 months. It involves exchanging interest at 10% on £20 million for interest at 6% on $30 million once a year. The term structure of risk-free interest rates in the United Kingdom is flat at 7% and the term structure of risk-free interest rates in the United States is flat at 4% (both with annual compounding). The current exchange rate (dollars per pound sterling) is 1.5500. What is the value of the swap to the party paying sterling? What is the value of the swap to the party paying dollars?

### Solution
The swap payments occur at **3 months** ($0.25$ years) and **15 months** ($1.25$ years).
We value the swap from the perspective of the party **paying sterling** (receives USD at 6%, pays GBP at 10%).

* **USD Bond ($B_D$)**:
 * Coupons: $6\% \times \$30\text{M} = \$1.8\text{M}$.
 * Cash flows: $\$1.8\text{M}$ at 0.25y, $\$31.8\text{M}$ at 1.25y (interest + principal).
 * Discount at U.S. risk-free rate of $4\%$ per annum (annual compounding).
* **GBP Bond ($B_{UK}$)**:
 * Coupons: $10\% \times \pounds 20\text{M} = \pounds 2\text{M}$.
 * Cash flows: $\pounds 2\text{M}$ at 0.25y, $\pounds 22\text{M}$ at 1.25y.
 * Discount at UK risk-free rate of $7\%$ per annum (annual compounding).
* **Swap Value**:
 $$V_{\text{swap}} = B_D - S_0 B_{UK}$$


In [11]:
# --- Question 7.4 Calculations ---
usd_princ_74 = 30.0
gbp_princ_74 = 20.0
usd_rate_74 = 0.06
gbp_rate_74 = 0.10

r_usd_74 = 0.04 # Annual compounding
r_gbp_74 = 0.07 # Annual compounding
spot_s0_74 = 1.5500 # USD per GBP

times_74 = np.array([0.25, 1.25])

# Dollar bond flows
usd_flows_74 = usd_rate_74 * usd_princ_74 * np.ones_like(times_74)
usd_flows_74[-1] += usd_princ_74
dfs_usd_74 = (1.0 + r_usd_74) ** (-times_74)
b_d_74 = (usd_flows_74 * dfs_usd_74).sum()

# Sterling bond flows
gbp_flows_74 = gbp_rate_74 * gbp_princ_74 * np.ones_like(times_74)
gbp_flows_74[-1] += gbp_princ_74
dfs_gbp_74 = (1.0 + r_gbp_74) ** (-times_74)
b_uk_74 = (gbp_flows_74 * dfs_gbp_74).sum()

# Value of Swap in USD (paying sterling = receives USD, pays GBP)
v_pay_gbp = b_d_74 - spot_s0_74 * b_uk_74

print(f"Value of USD Bond: ${b_d_74:.4f} Million USD")
print(f"Value of GBP Bond: £{b_uk_74:.4f} Million GBP")
print(f"Value to the party paying Sterling (in USD): ${v_pay_gbp:.4f} Million USD")
print(f"Value to the party paying Sterling (in GBP): £{v_pay_gbp / spot_s0_74:.4f} Million GBP")
print(f"Value to the party paying Dollars (in USD): +${-v_pay_gbp:.4f} Million USD")


Value of USD Bond: $32.0610 Million USD
Value of GBP Bond: £22.1823 Million GBP
Value to the party paying Sterling (in USD): $-2.3216 Million USD
Value to the party paying Sterling (in GBP): £-1.4978 Million GBP
Value to the party paying Dollars (in USD):  +$2.3216 Million USD


---
## Practice Question 7.5

### Question
Explain the difference between the credit risk and the market risk in a swap.

### Solution
* **Market Risk**: This is the exposure to movements in interest rates or exchange rates. If interest rates rise or fall, the present value of the swap's future payments will shift, turning the contract into a net asset or a net liability. Market risk is active every second and can be hedged by taking offsetting positions in bonds, futures, or other swaps.
* **Credit Risk**: This is the exposure to a counterparty's **default**. Unlike loans (where you are always at risk of losing the outstanding principal), credit risk in a swap only exists if **two conditions are met**:
 1. The counterparty defaults.
 2. The swap has a **positive value** (an asset) to you on the default date.
 
 If the swap has a negative value to you when the counterparty defaults, you don't lose anything (you simply settle the market value of the contract). Moreover, because interest rate swaps do not exchange principal, the potential credit loss is limited strictly to the net interest rate wiggles.


---
## Practice Question 7.6

### Question
A corporate treasurer tells you that he has just negotiated a five-year loan at a competitive fixed rate of interest of 5.2%. The treasurer explains that he achieved the 5.2% rate by borrowing at a six-month floating reference rate plus 150 basis points and swapping the floating reference rate for 3.7%. He goes on to say that this was possible because his company has a comparative advantage in the floating-rate market. What has the treasurer overlooked?

### Solution
The treasurer has overlooked **credit risk** and **credit spreads**!
1. **Net Borrowing Cost**: Borrowing floating at $\text{Floating} + 1.50\%$ and paying fixed $3.7\%$ in the swap indeed yields:
 $$\text{Net Cost} = (\text{Floating} + 1.50\%) + 3.70\% - \text{Floating} = 5.20\% \text{ fixed}$$
2. **The Illusion of Comparative Advantage**:
 * The $3.7\%$ fixed rate is the swap rate (based on highly-rated risk-free counterparties).
 * The $1.50\%$ (150 bps) floating premium is the treasurer's *own credit spread* reflecting their credit risk.
 * By combining the swap, the treasurer has not beaten the market; they have simply paid their own credit spread of $1.50\%$ over the risk-free swap rate ($3.7\%$).
 * Furthermore, the treasurer now carries **counterparty credit risk** from the swap intermediary bank, and **basis risk / rollover risk** (if the floating rate on the loan resets differently from the floating rate on the swap).


---
## Practice Question 7.7

### Question
A bank enters into an interest rate swap with a nonfinancial counterparty using bilaterally clearing where it is paying a fixed rate of 3% and receiving floating. No collateral is posted and no other transactions are outstanding between the bank and the counterparty. What credit risk is the bank subject to? Discuss whether the credit risk is greater when the yield curve is upward sloping or when it is downward sloping.

### Solution
1. **The Bank's Credit Risk**:
 The bank pays fixed $3\%$ and receives floating. The swap is an asset to the bank when **floating rates rise above $3\%$**. The bank faces credit risk if the counterparty defaults when the floating leg is higher than the fixed leg.
2. **Yield Curve Shapes**:
 * **Upward Sloping Yield Curve**: Forward rates increase in the future. The expected floating payments received by the bank will grow over time, meaning the swap is highly expected to have a **positive value** to the bank in its later years. This increases the bank's credit risk exposure.
 * **Downward Sloping Yield Curve**: Forward rates decrease in the future. The bank expects to receive lower floating rates than the $3\%$ it pays, meaning the swap is expected to be a **liability** (negative value) to the bank. If the counterparty defaults, the bank faces no loss.
 
Thus, the bank's credit risk is **significantly greater when the yield curve is upward sloping**.


---
## Practice Question 7.8

### Question
Companies X and Y have been offered the following rates per annum on a $5 million 10-year investment:
* **Company X**: Fixed $8.0\%$ | Floating $\text{LIBOR}$
* **Company Y**: Fixed $8.8\%$ | Floating $\text{LIBOR}$

Company X requires a fixed-rate investment; company Y requires a floating-rate investment. Design a swap that will net a bank, acting as intermediary, 0.2% per annum and will appear equally attractive to X and Y.

### Solution
*Note: This is an **investment** (receiving money), not a loan. For investments, companies want **higher** yields.*

1. **Identify Spreads**:
 * Fixed rate spread: $8.8\% - 8.0\% = 0.8\%$ (Company Y has a $0.8\%$ advantage in Fixed yields).
 * Floating rate spread: $\text{LIBOR} - \text{LIBOR} = 0\%$.
 * Total potential gain from swap: $0.8\% - 0\% = 0.8\%$ per annum.
2. **Allocate Gains**:
 * The intermediary bank receives $0.2\%$, leaving $0.8\% - 0.2\% = 0.6\%$.
 * Split equally, each company increases its investment return by $0.3\%$.
3. **Determine Target Yields**:
 * **Company X** wants Fixed: Offered $8.0\%$ fixed in the market. With $0.3\%$ gain, target return is **$8.3\%$** fixed.
 * **Company Y** wants Floating: Offered $\text{LIBOR}$ floating in the market. With $0.3\%$ gain, target return is **$\text{LIBOR} + 0.3\%$** floating.
4. **Swap Design**:
 * **Company X** invests at floating $\text{LIBOR}$ in the market.
 * Receives $\text{LIBOR}$ from market.
 * Receives fixed $8.3\%$ from bank.
 * Pays $\text{LIBOR}$ to bank.
 * **Net Return for X**: $\text{LIBOR} + 8.3\% - \text{LIBOR} = \mathbf{8.3\%}$ fixed (Gain of $0.3\%$, matches target!).
 * **Company Y** invests at fixed $8.8\%$ in the market.
 * Receives fixed $8.8\%$ from market.
 * Receives $\text{LIBOR}$ from bank.
 * Pays fixed $8.5\%$ to bank.
 * **Net Return for Y**: $8.8\% + \text{LIBOR} - 8.5\% = \mathbf{\text{LIBOR} + 0.3\%}$ floating (Gain of $0.3\%$, matches target!).
 * **The Bank** (Intermediary):
 * Receives $\text{LIBOR}$ from X, pays $\text{LIBOR}$ to Y (net floating flow is 0).
 * Receives fixed $8.5\%$ from Y, pays fixed $8.3\%$ to X.
 * **Bank Spread**: $8.5\% - 8.3\% = \mathbf{0.2\%}$ per annum (Matches target!).


---
## Practice Question 7.9

### Question
A financial institution has entered into an interest rate swap with company X. Under the terms of the swap, it receives 4% per annum and pays six-month LIBOR on a principal of $10 million for five years. Payments are made every six months. Suppose that company X defaults on the sixth payment date (end of year 3) when six-month forward LIBOR rates for all maturities are 2% per annum. What is the loss to the financial institution? Assume that six-month LIBOR was 3% per annum halfway through year 3 and that at the time of the default all OIS rates are 1.8% per annum. OIS rates are expressed with continuous compounding; other rates are expressed with semiannual compounding.

### Solution
1. **The Cash Flow on the Default Date (End of Year 3)**:
 * The floating rate was locked in 6 months ago at $3\%$ per annum (semiannual).
 * Floating CF paid: $0.5 \times 0.030 \times \$10\text{M} = \$0.150\text{M}$.
 * Fixed CF received: $0.5 \times 0.040 \times \$10\text{M} = \$0.200\text{M}$.
 * Net Cash Flow due at Year 3: $+0.200 - 0.150 = \$0.050\text{M}$. Since Company X defaults *on* this date, this payment is lost.
2. **Remaining Cash Flows (Years 3.5, 4.0, 4.5, 5.0)**:
 * 4 remaining semiannual payments.
 * Forward LIBOR rates are flat at $2.0\%$.
 * Floating CF: $0.5 \times 0.020 \times \$10\text{M} = \$0.100\text{M}$.
 * Fixed CF: $0.5 \times 0.040 \times \$10\text{M} = \$0.200\text{M}$.
 * Net CF: $+\$0.100\text{M}$ expected on each future date.
3. **Discounting**:
 * OIS Continuous rate is $1.8\%$.
 * PV of loss = $\$0.050\text{M} + \sum_{t \in \{0.5, 1.0, 1.5, 2.0\}} 0.100 e^{-0.018 \times t}$.


In [12]:
# --- Question 7.9 Calculations ---
notional_79 = 10.0
r_ois_79 = 0.018

# Lost flow today (End of Year 3)
loss_today = 0.5 * (0.04 - 0.03) * notional_79 # Fixed - Floating

# Future expected flows (Times relative to today)
times_79 = np.array([0.5, 1.0, 1.5, 2.0])
future_flows = 0.5 * (0.04 - 0.02) * notional_79 * np.ones_like(times_79)

dfs_79 = np.exp(-r_ois_79 * times_79)
pvs_79 = future_flows * dfs_79

total_loss_79 = loss_today + pvs_79.sum()
print(f"Loss Today: ${loss_today*1e6:,.2f}")
print(f"PV of Future Flows: ${pvs_79.sum()*1e6:,.2f}")
print(f"Total Credit Loss: ${total_loss_79*1e6:,.2f} USD")


Loss Today: $50,000.00
PV of Future Flows: $391,120.29
Total Credit Loss: $441,120.29 USD


---
## Practice Question 7.10

### Question
A financial institution has entered into a 10-year currency swap with company Y. Under the terms of the swap, the financial institution receives interest at 3% per annum in Swiss francs and pays interest at 8% per annum in U.S. dollars. Interest payments are exchanged once a year. The principal amounts are 7 million dollars and 10 million francs. Suppose that company Y declares bankruptcy at the end of year 6, when the exchange rate is $0.80 per franc. What is the cost to the financial institution? Assume that, at the end of year 6, risk-free interest rates are 3% per annum in Swiss francs and 8% per annum in U.S. dollars for all maturities. All interest rates are quoted with annual compounding.

### Solution
1. **Swap Valuation at Default (End of Year 6)**:
 * The swap has 4 years remaining.
 * **Swiss Franc Leg (Receive)**: The financial institution receives $3\%$ coupons and a $10\text{M CHF}$ principal. Since the CHF interest rate is flat at $3\%$ (annual compounding), the value of this CHF bond in CHF is exactly par:
 $$B_{\text{CHF}} = 10\text{ million CHF}$$
 * **U.S. Dollar Leg (Pay)**: The financial institution pays $8\%$ coupons and a $\$7\text{M USD}$ principal. Since the USD interest rate is flat at $8\%$ (annual compounding), the value of this USD bond in USD is exactly par:
 $$B_{\text{USD}} = 7\text{ million USD}$$
2. **Convert to Dollars at Spot Exchange Rate ($S = 0.80\text{ USD/CHF}$)**:
 * Value of received CHF bond in Dollars: $0.80 \times 10 = \$8\text{ million USD}$.
 * Value of paid USD bond in Dollars: $\$7\text{ million USD}$.
 * Net Swap Value to Bank:
 $$V_{\text{swap}} = S B_{\text{CHF}} - B_{\text{USD}} = 8 - 7 = +\$1.0\text{ million USD}$$
3. **The Loss**:
 * Since the swap is currently an asset to the bank (value is **$+\$1.0$ million**), the bankruptcy of Company Y results in a total loss of **$\$1.0$ million USD** to the financial institution.


---
## Practice Question 7.11

### Question
Companies A and B face the following interest rates (adjusted for the differential impact of taxes):
* **Company A**: USD Floating: $\text{Floating} + 0.5\%$ | CAD Fixed: $5.0\%$
* **Company B**: USD Floating: $\text{Floating} + 1.0\%$ | CAD Fixed: $6.5\%$

Assume that A wants to borrow U.S. dollars at a floating rate of interest and B wants to borrow Canadian dollars at a fixed rate of interest. A financial institution is planning to arrange a swap and requires a 50-basis-point spread. If the swap is equally attractive to A and B, what rates of interest will A and B end up paying?

### Solution
1. **Identify Spreads**:
 * CAD Fixed spread: $6.5\% - 5.0\% = 1.5\%$ (Company A has a $1.5\%$ advantage).
 * USD Floating spread: $(\text{Floating} + 1.0\%) - (\text{Floating} + 0.5\%) = 0.5\%$ (Company A has a $0.5\%$ advantage).
 * Total potential gain from swap: $1.5\% - 0.5\% = 1.0\%$ per annum.
2. **Allocate Gains**:
 * The intermediary financial institution takes $50\text{ bps} = 0.5\%$, leaving $1.0\% - 0.5\% = 0.5\%$.
 * Split equally, each company saves $0.25\%$.
3. **Determine Target Costs**:
 * **Company A** wants USD floating: Offered $\text{Floating} + 0.5\%$. With $0.25\%$ savings, net cost is **$\text{Floating} + 0.25\%$** in U.S. Dollars.
 * **Company B** wants CAD fixed: Offered $6.5\%$. With $0.25\%$ savings, net cost is **$6.25\%$** fixed in Canadian Dollars.


---
## Practice Question 7.12

### Question
After it hedges its foreign exchange risk using forward contracts, is the financial institution’s average spread in Figure 7.11 likely to be greater than or less than 20 basis points? Explain your answer.

### Solution
It is likely to be **less than 20 basis points**. 
* **The Net Spread**: In Figure 7.11, the financial institution receives interest at $1.1\%$ in Japanese Yen and pays interest at $0.9\%$ in U.S. Dollars. This is a nominal spread of $+0.2\% = 20\text{ basis points}$.
* **The Forward Premium**: Japanese interest rates are lower than U.S. interest rates, which means Jpy trades at a **forward premium** relative to USD (meaning the Yen is expected to appreciate over time). 
* **The Hedges**: The bank receives Yen and pays Dollars. To hedge its exchange risk, the bank sells Yen interest payments forward and buys Yen forward to fund the final principal exchange (since it returns Yen principal and receives USD principal).
* **The Asymmetry**: Because Yen is at a forward premium, the final Yen principal payment that the bank must pay in Year 3 is **much more expensive in USD terms** than at spot. The net impact of locking in these forward exchange rates significantly reduces the net USD value of the spread, making the final hedged average spread **less than 20 basis points**.


---
## Practice Question 7.13

### Question
“Nonfinancial companies with high credit risks are the ones that cannot access fixed-rate markets directly. They are the companies that are most likely to be paying fixed and receiving floating in an interest rate swap.” Assume that this statement is true. Do you think it increases or decreases the risk of a financial institution’s swap portfolio? Assume that companies are most likely to default when interest rates are high.

### Solution
This **decreases** the credit risk of the financial institution's swap portfolio!
1. **The Bank's Exposure**:
 * If these high-credit-risk counterparties are **paying fixed and receiving floating**, then the financial institution is **paying floating and receiving fixed**.
 * The swap is an asset to the bank (positive value) only when **interest rates are low** (bank pays low floating, receives high fixed).
2. **Default Timing**:
 * According to the assumption, companies default when **interest rates are high**.
 * When interest rates are high, the bank is paying high floating and receiving low fixed. The swap has a **negative value** (a liability) to the bank.
 * If the counterparty defaults when the swap is a liability to the bank, the bank faces **zero credit loss** (the exposure is zero).
 
Therefore, because defaults occur when the bank's exposure is negative, this natural alignment dramatically **reduces** credit risk.


---
## Practice Question 7.14

### Question
Why is the expected loss to a bank from a default on a swap with a counterparty less than the expected loss from the default on a loan to the counterparty when the loan and swap have the same principal? Assume that there are no other derivatives transactions between the bank and the counterparty, that the swap is cleared bilaterally, and that no collateral is provided by the counterparty in the case of either the swap or the loan.

### Solution
1. **No Principal Exchange**: In a standard interest rate swap, the principal is purely a fictional "notional" amount used only to calculate interest payments. In a default, the bank only loses the interest payment differential. In a loan, the entire outstanding principal is at risk of being lost.
2. **Two-Way Asset/Liability Structure**: A loan is always an asset to the bank (exposure is always positive). A swap is a bilateral agreement that wiggles between being an asset (positive value) and a liability (negative value). If the counterparty defaults when the swap is a liability to the bank, the bank experiences **no credit loss**. Thus, the credit exposure on a swap is only active about $50\%$ of the time.


---
## Practice Question 7.15

### Question
A bank finds that its assets are not matched with its liabilities. It is taking floating-rate deposits and making fixed-rate loans. How can swaps be used to offset the risk?

### Solution
The bank is currently:
* Receiving fixed interest (from fixed-rate loans).
* Paying floating interest (to floating-rate deposits).

This creates huge exposure to **rising interest rates** (if rates spike, deposit costs will exceed loan income).

**To hedge this risk**:
The bank should enter into an interest rate swap where it **pays fixed** and **receives floating**:
* **The Floating Match**: The floating reference rate received in the swap will match and offset the floating interest paid to depositor liabilities.
* **The Fixed Match**: The fixed rate paid on the swap will be funded by the fixed interest received from loan assets.
* **The Result**: The bank locks in a completely secure, fixed-interest spread, eliminating all interest rate risk.


---
## Practice Question 7.16

### Question
Explain how you would value a swap that is the exchange of a floating rate in one currency for a fixed rate in another currency.

### Solution
This is a **Fixed-for-Floating Currency Swap**. It can be valued in two ways:
1. **Portfolio of Bonds**:
 The swap is treated as a long position in a bond in the received currency, and a short position in a bond in the paid currency:
 $$V_{\text{swap}} = S_0 B_{\text{foreign}} - B_{\text{domestic}}$$
 * One bond will be a fixed-rate bond.
 * The other bond will be a floating-rate bond (valued using standard swap-float methods where it resets to par at payment dates).
2. **Portfolio of Forward Contracts**:
 * Use the floating currency's forward interest rate curve to project all the future floating rate payments in their native currency.
 * Discount these floating cash flows in their native currency at the floating-currency risk-free rate.
 * Discount the fixed cash flows in their native currency at the fixed-currency risk-free rate.
 * Convert the foreign-denominated leg value to domestic terms using the spot exchange rate, and find the net difference.


---
## Practice Question 7.17

### Question
OIS rates are 3.4% for all maturities. What is the value of an OIS swap with two years to maturity where 3% is received and the floating reference rate is paid. Assume annual compounding, annual payments, and $100 million principal.

### Solution
1. **Convert Discount Rate**: OIS risk-free rate is flat at $3.4\%$ continuous.
 Annually compounded risk-free rate $R_a$:
 $$R_a = e^{0.034} - 1 = 3.4583\%$$
2. **Floating Leg ($B_{\text{floating}}$)**:
 Since the floating payments wiggle in line with the OIS zero curve, the OIS floating bond is worth exactly par immediately after payments:
 $$B_{\text{floating}} = \$100\text{ million}$$
3. **Fixed Leg ($B_{\text{fixed}}$)**:
 Annual coupons are $3\% \times \$100\text{M} = \$3\text{M}$.
 * Year 1 CF: $\$3\text{M}$
 * Year 2 CF: $\$103\text{M}$
 * Discounted PV: $3 e^{-0.034 \times 1} + 103 e^{-0.034 \times 2}$.
4. **Swap Value** (receives fixed, pays floating):
 $$V = B_{\text{fixed}} - B_{\text{floating}}$$


In [13]:
# --- Question 7.17 Calculations ---
notional_717 = 100.0
fixed_rate_717 = 0.03
r_ois_717 = 0.034 # Continuous

times_717 = np.array([1.0, 2.0])
fixed_flows = fixed_rate_717 * notional_717 * np.ones_like(times_717)
fixed_flows[-1] += notional_717

dfs_717 = np.exp(-r_ois_717 * times_717)
b_fixed_717 = (fixed_flows * dfs_717).sum()

# Floating leg value is par
b_float_717 = notional_717

v_swap_717 = b_fixed_717 - b_float_717

print(f"Value of Fixed Leg Bond: ${b_fixed_717:.4f} Million")
print(f"Value of Swap (V_swap): ${v_swap_717:.4f} Million USD")
print(f"Value of Swap (exact): ${v_swap_717*1e6:,.2f} USD")


Value of Fixed Leg Bond: $99.1285 Million
Value of Swap (V_swap): $-0.8715 Million USD
Value of Swap (exact): $-871,456.71 USD


---
## Practice Question 7.18

### Question
A financial institution has entered into a swap where it agreed to make quarterly payments at a rate of 3% per annum and receive the SOFR three-month reference rate on a notional principal of $100 million. The swap now has a remaining life of 7.5 months. Assume the risk-free rates with continuous compounding (calculated from SOFR) for 1.5, 4.5, and 7.5 months are 2.8%, 3.0%, and 3.1%, respectively. Assume also that the continuously compounded risk-free rate observed for the last 1.5 months is 2.7%. Estimate the value of the swap.

### Solution
Swap payments occur quarterly ($\Delta t = 0.25$ years) at $1.5$, $4.5$, and $7.5$ months ($0.125$, $0.375$, and $0.625$ years).

1. **Floating Rate for First period (Maturity 1.5m)**:
 * Since this is a quarterly exchange, the period started 1.5 months ago.
 * The first payment is a **blend**: $50\%$ observed rate ($2.7\%$), $50\%$ zero rate ($2.8\%$).
 * Blended continuous rate: $0.5 \times 2.7\% + 0.5 \times 2.8\% = 2.75\%$ continuous.
 * Semiannual/Quarterly equivalent: $R_m = 4(e^{0.0275/4} - 1) = 2.7595\%$.
 * Floating CF: $0.25 \times 0.027595 \times \$100\text{M} = \$0.6899\text{M}$.
2. **Floating Rate for Second period (Maturity 4.5m)**:
 * Forward rate between 1.5m and 4.5m:
 $$F = \frac{0.030 \times 0.375 - 0.028 \times 0.125}{0.375 - 0.125} = 3.10\% \text{ continuous}$$
 * Quarterly equivalent: $R_m = 4(e^{0.031/4}-1) = 3.112\%$.
 * Floating CF: $0.25 \times 0.03112 \times \$100\text{M} = \$0.7780\text{M}$.
3. **Floating Rate for Third period (Maturity 7.5m)**:
 * Forward rate between 4.5m and 7.5m:
 $$F = \frac{0.031 \times 0.625 - 0.030 \times 0.375}{0.625 - 0.375} = 3.25\% \text{ continuous}$$
 * Quarterly equivalent: $R_m = 4(e^{0.0325/4}-1) = 3.263\%$.
 * Floating CF: $0.25 \times 0.03263 \times \$100\text{M} = \$0.8158\text{M}$.
4. **Fixed Cash Flows**:
 * $-0.25 \times 0.03 \times \$100\text{M} = -\$0.750\text{M}$ at each date.


In [14]:
# --- Question 7.18 Calculations ---
notional_718 = 100.0
fixed_rate_718 = 0.03
times_718 = np.array([1.5/12.0, 4.5/12.0, 7.5/12.0])
rates_718 = np.array([0.028, 0.030, 0.031]) # Continuous risk-free rates

# 1. Calculate Continuous Floating Rates (Forward Rates)
rc_025 = 0.5 * 0.027 + 0.5 * 0.028 # Blend for first payment
rc_075 = (rates_718[1]*times_718[1] - rates_718[0]*times_718[0]) / (times_718[1] - times_718[0])
rc_125 = (rates_718[2]*times_718[2] - rates_718[1]*times_718[1]) / (times_718[2] - times_718[1])
rc_rates_718 = np.array([rc_025, rc_075, rc_125])

# 2. Convert to Quarterly compounding
rm_rates_718 = 4 * (np.exp(rc_rates_718 / 4) - 1)

# 3. Cash flows (pays fixed, receives floating)
fixed_cfs_718 = -0.25 * fixed_rate_718 * notional_718 * np.ones_like(times_718)
floating_cfs_718 = 0.25 * rm_rates_718 * notional_718
net_cfs_718 = fixed_cfs_718 + floating_cfs_718 # Receive Floating, Pay Fixed

dfs_718 = np.exp(-rates_718 * times_718)
pvs_718 = net_cfs_718 * dfs_718
v_swap_718 = pvs_718.sum()

print("Maturity Cash Flows:")
for t, f_cf, fl_cf, net, pv in zip(times_718, fixed_cfs_718, floating_cfs_718, net_cfs_718, pvs_718):
 print(f" t={t:.3f}y -> Fixed: {f_cf:.4f}M | Float: {fl_cf:.4f}M | Net: {net:.4f}M | PV: {pv:.4f}M")
print(f"\nEstimated Value of the Swap to party receiving floating: ${v_swap_718*1e6:,.2f} USD")


Maturity Cash Flows:
  t=0.125y -> Fixed: -0.7500M | Float: 0.6899M | Net: -0.0601M | PV: -0.0599M
  t=0.375y -> Fixed: -0.7500M | Float: 0.7780M | Net: 0.0280M | PV: 0.0277M
  t=0.625y -> Fixed: -0.7500M | Float: 0.8158M | Net: 0.0658M | PV: 0.0645M

Estimated Value of the Swap to party receiving floating: $32,323.29 USD


---
## Practice Question 7.19

### Question
(a) Company A has been offered the swap quotes in Table 7.4. It can borrow for three years at 3.45%. What floating rate can it swap this fixed rate into? 
(b) Company B has been offered the swap quotes in Table 7.4. It can borrow for five years at floating plus 75 basis points. What fixed rate can it swap this rate into? 
(c) Explain the rollover risks that Company B is taking.

### Solution
*Assuming Table 7.4 has standard swap rate quotes (e.g. 3-year swap rate = $3.20\% / 3.24\%$ and 5-year swap rate = $3.50\% / 3.54\%$)*:

* **(a) Company A** (wants to swap its $3.45\%$ fixed borrowing into floating):
 * Company A pays fixed $3.45\%$ to its loan lenders.
 * In the swap, Company A receives fixed at the **bid rate** of the swap dealer ($3.20\%$) and pays floating ($\text{SOFR}$).
 * **Net Cost for A**:
 $$\text{Net Cost} = 3.45\% + \text{SOFR} - 3.20\% = \mathbf{\text{SOFR} + 0.25\%}$$
* **(b) Company B** (wants to swap its $\text{Floating} + 0.75\%$ borrowing into fixed):
 * Company B pays floating $\text{SOFR} + 0.75\%$ to its loan lenders.
 * In the swap, Company B pays fixed at the **ask rate** of the swap dealer ($3.54\%$) and receives floating ($\text{SOFR}$).
 * **Net Cost for B**:
 $$\text{Net Cost} = (\text{SOFR} + 0.75\%) + 3.54\% - \text{SOFR} = \mathbf{4.29\% \text{ fixed}}$$
* **(c) Rollover Risk**:
 * Company B's net fixed rate of $4.29\%$ assumes that its floating credit spread over $\text{SOFR}$ remains flat at $75\text{ bps}$ over the entire 5 years.
 * In practice, if Company B's credit rating deteriorates or bank lending tightens, and B has to renew or roll over its floating-rate debt, the bank may increase its floating spread to (say) $150\text{ bps}$.
 * The swap *only* hedges the benchmark $\text{SOFR}$ rate, not B's credit risk. B's net borrowing cost would rise to $3.54\% + 1.50\% = 5.04\%$. This risk of increasing credit spreads during debt rollovers is **rollover risk**.


---
## Practice Question 7.20

### Question
The one-year LIBOR rate is 3%, and the LIBOR forward rate for the 1- to 2-year period is 3.2%. The three-year swap rate for a swap with annual payments is 3.2%. What is the LIBOR forward rate for the 2- to 3-year period if OIS zero rates for maturities of one, two, and three years are 2.5%, 2.7%, and 2.9%, respectively. What is the value of a three-year swap where 4% is received and LIBOR is paid on a principal of $100 million. All rates are annually compounded.

### Solution
1. **OIS Discount Factors** (annual compounding):
 * $DF_1 = \frac{1}{1.025} = 0.97561$
 * $DF_2 = \frac{1}{(1.027)^2} = 0.94813$
 * $DF_3 = \frac{1}{(1.029)^3} = 0.91773$
2. **Find LIBOR Forward Rate for Year 3 ($L_3$)**:
 Since the 3-year swap rate is $3.20\%$, a swap paying LIBOR and receiving $3.20\%$ has a value of exactly 0 at inception:
 $$0.032(DF_1 + DF_2 + DF_3) = L_1 DF_1 + L_2 DF_2 + L_3 DF_3$$
 
 Using $L_1 = 3.0\%$ and $L_2 = 3.2\%$, we solve for $L_3$:
 $$0.032(0.97561 + 0.94813 + 0.91773) = 0.030(0.97561) + 0.032(0.94813) + L_3(0.91773)$$
 $$0.090927 = 0.059609 + L_3(0.91773) \implies L_3 = 3.4126\%$$
3. **Value of 3-Year Swap receiving 4% and paying LIBOR**:
 Since a swap receiving $3.2\%$ has a value of 0, receiving $4\%$ is equivalent to receiving an extra $0.8\%$ annually:
 $$V = (4\% - 3.2\%) \times (DF_1 + DF_2 + DF_3) \times \$100\text{M}$$


In [15]:
# --- Question 7.20 Calculations ---
notional_720 = 100.0

r_ois = np.array([0.025, 0.027, 0.029])
dfs_720 = 1.0 / ((1.0 + r_ois) ** np.array([1.0, 2.0, 3.0]))

# Forward LIBOR rates
L1 = 0.03
L2 = 0.032
swap_rate_720 = 0.032

# Solve for L3: swap_rate * sum(DF) = L1*DF1 + L2*DF2 + L3*DF3
L3 = (swap_rate_720 * dfs_720.sum() - (L1*dfs_720[0] + L2*dfs_720[1])) / dfs_720[2]

# Value of 4% receive swap
v_swap_720 = (0.04 - swap_rate_720) * dfs_720.sum() * notional_720

print(f"OIS Discount Factors: {dfs_720}")
print(f"LIBOR Forward Rate for Year 3 (L3): {L3*100:.4f}%")
print(f"Value of 3-year Swap receiving 4%: ${v_swap_720:.4f} Million USD")
print(f"Value of 3-year Swap receiving 4% (exact): ${v_swap_720*1e6:,.2f} USD")


OIS Discount Factors: [0.97560976 0.94811084 0.9178123 ]
LIBOR Forward Rate for Year 3 (L3): 3.4126%
Value of 3-year Swap receiving 4%: $2.2732 Million USD
Value of 3-year Swap receiving 4% (exact): $2,273,226.32 USD


---
## Practice Question 7.21

### Question
A financial institution has entered into a swap where it agreed to receive quarterly payments at a rate of 2% per annum and pay the SOFR three-month reference rate on a notional principal of $100 million. The swap now has a remaining life of 10 months. Assume the risk-free rates with continuous compounding (calculated from SOFR) for 1 month, 4 months, 7 months, and 10 months are 1.4%, 1.6%, 1.7%, and 1.8%, respectively. Assume also that the continuously compounded risk-free rate observed for the last two months is 1.1%. Estimate the value of the swap.

### Solution
Swap payments occur quarterly ($\Delta t = 0.25$ years) at $1$, $4$, $7$, and $10$ months ($0.0833$, $0.3333$, $0.5833$, and $0.8333$ years).

1. **Floating Rate for First period (Maturity 1m)**:
 * Since this is a quarterly exchange, the period started 2 months ago.
 * Blended rate: $2/3$ observed rate ($1.1\%$), $1/3$ zero rate ($1.4\%$).
 * Blended continuous rate: $\frac{2}{3} \times 1.1\% + \frac{1}{3} \times 1.4\% = 1.20\%$ continuous.
 * Quarterly equivalent: $R_m = 4(e^{0.0120/4} - 1) = 1.2018\%$.
 * Floating CF paid: $0.25 \times 0.012018 \times \$100\text{M} = \$0.3005\text{M}$.
2. **Floating Rate for Second period (Maturity 4m)**:
 * Forward rate between 1m (0.0833y) and 4m (0.3333y):
 $$F = \frac{0.016 \times 0.3333 - 0.014 \times 0.0833}{0.3333 - 0.0833} = 1.667\% \text{ continuous}$$
 * Quarterly equivalent: $R_m = 4(e^{0.01667/4}-1) = 1.670\%$.
 * Floating CF paid: $0.25 \times 0.01670 \times \$100\text{M} = \$0.4176\text{M}$.
3. **Floating Rate for Third period (Maturity 7m)**:
 * Forward rate between 4m (0.3333y) and 7m (0.5833y):
 $$F = \frac{0.017 \times 0.5833 - 0.016 \times 0.3333}{0.5833 - 0.3333} = 1.833\% \text{ continuous}$$
 * Quarterly equivalent: $R_m = 4(e^{0.01833/4}-1) = 1.838\%$.
 * Floating CF paid: $0.25 \times 0.01838 \times \$100\text{M} = \$0.4594\text{M}$.
4. **Floating Rate for Fourth period (Maturity 10m)**:
 * Forward rate between 7m (0.5833y) and 10m (0.8333y):
 $$F = \frac{0.018 \times 0.8333 - 0.017 \times 0.5833}{0.8333 - 0.5833} = 2.033\% \text{ continuous}$$
 * Quarterly equivalent: $R_m = 4(e^{0.02033/4}-1) = 2.038\%$.
 * Floating CF paid: $0.25 \times 0.02038 \times \$100\text{M} = \$0.5096\text{M}$.
5. **Fixed Cash Flows**:
 * Receives $+0.25 \times 0.02 \times \$100\text{M} = +\$0.500\text{M}$ at each date.


In [16]:
# --- Question 7.21 Calculations ---
notional_721 = 100.0
fixed_rate_721 = 0.02
times_721 = np.array([1.0/12.0, 4.0/12.0, 7.0/12.0, 10.0/12.0])
rates_721 = np.array([0.014, 0.016, 0.017, 0.018])

# 1. Calculate Continuous Floating Rates (Forward Rates)
rc_1 = (2.0/3.0)*0.011 + (1.0/3.0)*0.014 # Blend
rc_4 = (rates_721[1]*times_721[1] - rates_721[0]*times_721[0]) / (times_721[1] - times_721[0])
rc_7 = (rates_721[2]*times_721[2] - rates_721[1]*times_721[1]) / (times_721[2] - times_721[1])
rc_10 = (rates_721[3]*times_721[3] - rates_721[2]*times_721[2]) / (times_721[3] - times_721[2])
rc_rates_721 = np.array([rc_1, rc_4, rc_7, rc_10])

# 2. Convert to Quarterly compounding
rm_rates_721 = 4 * (np.exp(rc_rates_721 / 4) - 1)

# 3. Cash flows (receives fixed, pays floating)
fixed_cfs_721 = 0.25 * fixed_rate_721 * notional_721 * np.ones_like(times_721)
floating_cfs_721 = 0.25 * rm_rates_721 * notional_721
net_cfs_721 = fixed_cfs_721 - floating_cfs_721

dfs_721 = np.exp(-rates_721 * times_721)
pvs_721 = net_cfs_721 * dfs_721
v_swap_721 = pvs_721.sum()

print("Maturity Cash Flows:")
for t, f_cf, fl_cf, net, pv in zip(times_721, fixed_cfs_721, floating_cfs_721, net_cfs_721, pvs_721):
 print(f" t={t:.3f}y -> Fixed: {f_cf:.4f}M | Float: {fl_cf:.4f}M | Net: {net:.4f}M | PV: {pv:.4f}M")
print(f"\nEstimated Value of the Swap to party receiving fixed: ${v_swap_721*1e6:,.2f} USD")


Maturity Cash Flows:
  t=0.083y -> Fixed: 0.5000M | Float: 0.3005M | Net: 0.1995M | PV: 0.1993M
  t=0.333y -> Fixed: 0.5000M | Float: 0.4175M | Net: 0.0825M | PV: 0.0820M
  t=0.583y -> Fixed: 0.5000M | Float: 0.4594M | Net: 0.0406M | PV: 0.0402M
  t=0.833y -> Fixed: 0.5000M | Float: 0.5096M | Net: -0.0096M | PV: -0.0095M

Estimated Value of the Swap to party receiving fixed: $312,072.05 USD


---
## Practice Question 7.22

### Question
Company A, a British manufacturer, wishes to borrow U.S. dollars at a fixed rate of interest. Company B, a U.S. multinational, wishes to borrow sterling at a fixed rate of interest. They have been quoted the following rates per annum (adjusted for differential tax effects):
* **Company A**: Sterling $11.0\%$ | U.S. Dollars $7.0\%$
* **Company B**: Sterling $10.6\%$ | U.S. Dollars $6.2\%$

Design a swap that will net a bank, acting as intermediary, 10 basis points per annum and that will produce a gain of 15 basis points per annum for each of the two companies.

### Solution
1. **Identify Spreads**:
 * Sterling Fixed spread: $11.0\% - 10.6\% = 0.4\%$ (Company B has a $0.4\%$ advantage).
 * Dollar Fixed spread: $7.0\% - 6.2\% = 0.8\%$ (Company B has a $0.8\%$ advantage).
 * Total potential gain from swap: $\text{Dollar spread} - \text{Sterling spread} = 0.8\% - 0.4\% = 0.4\% = 40\text{ bps}$.
2. **Allocate Gains**:
 * Intermediary bank gets $10\text{ bps} = 0.1\%$.
 * Company A and B get $15\text{ bps} = 0.15\%$ savings each.
 * Total gain allocated: $0.15\% + 0.15\% + 0.10\% = 0.40\%$. Perfect!
3. **Determine Target Costs**:
 * **Company A** wants USD: Offered $7.0\%$. With $0.15\%$ savings, target cost is **$6.85\%$** fixed in USD.
 * **Company B** wants Sterling: Offered $10.6\%$. With $0.15\%$ savings, target cost is **$10.45\%$** fixed in GBP.
4. **Swap Design**:
 * **Company A** borrows Sterling fixed in the market at $11.0\%$.
 * Pays GBP fixed $11.0\%$ to market.
 * Pays USD fixed $6.85\%$ to bank.
 * Receives GBP fixed $11.0\%$ from bank.
 * **Net USD Cost to A**: $6.85\%$ USD fixed (Savings of $0.15\%$, matches target!).
 * **Company B** borrows Dollars fixed in the market at $6.2\%$.
 * Pays USD fixed $6.2\%$ to market.
 * Pays GBP fixed $10.45\%$ to bank.
 * Receives USD fixed $6.2\%$ from bank.
 * **Net GBP Cost to B**: $10.45\%$ GBP fixed (Savings of $0.15\%$, matches target!).
 * **The Bank** (Intermediary):
 * Dollar flows: Receives $6.85\%$ from A, pays $6.2\%$ to B. **Net USD Flow**: $+0.65\%$ per annum.
 * Sterling flows: Receives $10.45\%$ from B, pays $11.0\%$ to A. **Net GBP Flow**: $-0.55\%$.
 * Assuming equivalent principal amounts, the bank nets $0.65\%\text{ USD} - 0.55\%\text{ GBP}$, which corresponds to a net gain of **$0.10\% = 10\text{ basis points}$** per annum.


---
## Practice Question 7.23

### Question
The five-year swap rate when cash flows are exchanged semiannually is 4%. A company wants a swap where it receives payments at 4.2% per annum on a notional principal of $10 million. The OIS zero curve is flat at 3.6%. How much should a derivatives dealer charge the company. Assume that all rates are expressed with semiannual compounding and ignore bid–ask spreads.

### Solution
1. **Analyze payments**:
 * The fair swap rate is $4.0\%$ per annum (semiannual).
 * The company wants to receive an off-market fixed rate of $4.2\%$, which is an extra $0.2\%$ per annum.
 * Extra cash flow received at each semiannual date: $0.5 \times 0.2\% \times \$10\text{M} = \$10,000$.
 * Number of payments over 5 years: $5 \times 2 = 10$ semiannual periods.
2. **Discount rate**:
 * OIS zero curve is flat at $3.6\%$ per annum with semiannual compounding.
 * The semiannual discount rate is $3.6\% / 2 = 1.8\%$ per period.
3. **Valuation**:
 * The dealer should charge the company an upfront fee equal to the present value of these extra cash flows:
 $$\text{Upfront Charge} = \sum_{t=1}^{10} \frac{\$10,000}{(1.018)^t}$$


In [17]:
# --- Question 7.23 Calculations ---
notional_723 = 10.0 # $10 Million
extra_payment = 0.5 * (0.042 - 0.040) * notional_723 # Fixed extra payment per semiannual period
r_ois_semiannual = 0.036 / 2.0 # 1.8% per period

n_periods = 10
periods = np.arange(1, n_periods + 1)
dfs_723 = (1.0 + r_ois_semiannual) ** (-periods)
upfront_charge = extra_payment * dfs_723.sum()

print(f"Extra Semiannual Payment: ${extra_payment*1e6:,.2f} USD")
print(f"Derivatives Dealer Upfront Charge: ${upfront_charge*1e6:,.2f} USD")


Extra Semiannual Payment: $10,000.00 USD
Derivatives Dealer Upfront Charge: $90,773.11 USD
